# Testes de Linearidade contra Alternativas Nao-Lineares

Neste notebook aplicamos testes formais de linearidade para decidir se um modelo
nao-linear (TAR ou STAR) e necessario e, em caso positivo, qual tipo usar.

## Conteudo
1. Teste LM de Luukkonen, Saikkonen e Terasvirta (1988)
2. Teste de Tsay (1989) para TAR
3. Teste de Hansen (1996) com bootstrap
4. Sequencia de testes de Terasvirta (1994): LSTAR vs ESTAR
5. Poder dos testes - simulacao Monte Carlo

## Referencias
- Luukkonen, R., Saikkonen, P. & Terasvirta, T. (1988). Testing linearity against smooth transition autoregressive models. *Biometrika*, 75, 491-499.
- Tsay, R.S. (1989). Testing and modeling threshold autoregressive processes. *Journal of the American Statistical Association*, 84, 231-240.
- Terasvirta, T. (1994). Specification, estimation, and evaluation of smooth transition autoregressive models. *JASA*, 89, 208-218.
- Hansen, B.E. (1996). Inference when a nuisance parameter is not identified under the null hypothesis. *Econometrica*, 64, 413-430.
- Tong, H. (1978). *On a threshold model*. Pattern Recognition and Signal Processing.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from archbox.threshold import (
    hansen_threshold_test,
    linearity_test,
    transition_type_test,
    tsay_test,
)

# Configuracao de graficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

# Carregar ambos os datasets
gdp = pd.read_csv('../data/us_gdp_growth.csv', parse_dates=['date'], index_col='date')
sp500 = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')

y_gdp = gdp['y'].values
y_sp = sp500['y'].values

print(f'US GDP Growth: {len(y_gdp)} obs')
print(f'S&P500 Returns: {len(y_sp)} obs')

## 1. Teste LM de Luukkonen, Saikkonen e Terasvirta (1988)

O teste **LM-STAR** testa linearidade contra a alternativa STAR.
Como o parametro $\gamma$ nao esta identificado sob $H_0$ (modelo linear),
usa-se uma **expansao de Taylor** de $G(s_t; \gamma, c)$ em torno de $\gamma = 0$.

A regressao auxiliar e:
$$
y_t = \beta_0 + \sum_{j=1}^{p} \beta_j y_{t-j} + \sum_{j=1}^{p} \delta_j y_{t-j} s_t + \sum_{j=1}^{p} \eta_j y_{t-j} s_t^2 + \sum_{j=1}^{p} \psi_j y_{t-j} s_t^3 + e_t
$$

As hipoteses sao:
- $H_0$: $\delta_j = \eta_j = \psi_j = 0$ para todo $j$ (linearidade)
- $H_1$: pelo menos um coeficiente e diferente de zero (STAR)

A estatistica e um teste F com distribuicao $F(3p, T - 4p - 1)$ sob $H_0$.

In [ ]:
# TODO: Implemente e execute o teste LM-STAR

print('=== Teste LM-STAR (Luukkonen, Saikkonen & Terasvirta, 1988) ===')
print()

# Teste para GDP Growth
lm_gdp = linearity_test(y_gdp, order=1, delay=1)
print('--- US GDP Growth ---')
print(f'Estatistica F: {lm_gdp.statistic:.4f}')
print(f'P-valor: {lm_gdp.pvalue:.4f}')
print(f'Teste: {lm_gdp.test_name}')
if lm_gdp.pvalue < 0.05:
    print('=> Rejeita H0 a 5%: evidencia de nao-linearidade tipo STAR')
else:
    print('=> Nao rejeita H0 a 5%: sem evidencia de nao-linearidade STAR')
print()

# Teste para S&P500 Returns
lm_sp = linearity_test(y_sp, order=1, delay=1)
print('--- S&P500 Returns ---')
print(f'Estatistica F: {lm_sp.statistic:.4f}')
print(f'P-valor: {lm_sp.pvalue:.4f}')
if lm_sp.pvalue < 0.05:
    print('=> Rejeita H0 a 5%: evidencia de nao-linearidade tipo STAR')
else:
    print('=> Nao rejeita H0 a 5%: sem evidencia de nao-linearidade STAR')

## 2. Teste de Tsay (1989)

O teste de **Tsay** e um teste F para a alternativa TAR (threshold abrupto).
Baseia-se na ideia de **autorregressao arranjada** (arranged autoregression):

1. Ordena-se as observacoes pela variavel de threshold $s_t = y_{t-d}$
2. Ajusta-se um AR recursivo (como CUSUM)
3. Testa-se se os residuos recursivos dependem de $s_t$

- $H_0$: Modelo linear AR
- $H_1$: Modelo TAR (existencia de threshold)

A estatistica segue uma distribuicao $F$ sob $H_0$.

In [ ]:
# TODO: Execute teste de Tsay

print('=== Teste de Tsay (1989) ===')
print()

# Teste para GDP Growth
tsay_gdp = tsay_test(y_gdp, order=1, delay=1)
print('--- US GDP Growth ---')
print(f'Estatistica F: {tsay_gdp.statistic:.4f}')
print(f'P-valor: {tsay_gdp.pvalue:.4f}')
print(f'Teste: {tsay_gdp.test_name}')
if tsay_gdp.pvalue < 0.05:
    print('=> Rejeita H0 a 5%: evidencia de threshold (TAR)')
else:
    print('=> Nao rejeita H0 a 5%: sem evidencia de threshold')
print()

# Teste para S&P500 Returns
tsay_sp = tsay_test(y_sp, order=1, delay=1)
print('--- S&P500 Returns ---')
print(f'Estatistica F: {tsay_sp.statistic:.4f}')
print(f'P-valor: {tsay_sp.pvalue:.4f}')
if tsay_sp.pvalue < 0.05:
    print('=> Rejeita H0 a 5%: evidencia de threshold (TAR)')
else:
    print('=> Nao rejeita H0 a 5%: sem evidencia de threshold')

## 3. Teste de Hansen (1996) com bootstrap

O teste de **Hansen** usa uma estatistica **sup-LM** com p-valor por **bootstrap**.
Isto resolve o problema do parametro nuisance ($c$) nao identificado sob $H_0$.

Procedimento:
1. Para cada $c$ no grid, calcular $\text{LM}(c)$
2. Estatistica do teste: $\text{sup-LM} = \sup_c \text{LM}(c)$
3. Gerar $B$ amostras bootstrap sob $H_0$ (AR linear)
4. Para cada amostra, calcular $\text{sup-LM}^*$
5. P-valor: proporcao de $\text{sup-LM}^* > \text{sup-LM}_{obs}$

Tipicamente usa-se $B = 1000$ ou mais replicas.

In [ ]:
# TODO: Execute teste de Hansen com 1000 replicas bootstrap

print('=== Teste de Hansen (1996) - Bootstrap ===')
print('(Isto pode levar alguns segundos...)')
print()

# GDP Growth
hansen_gdp = hansen_threshold_test(y_gdp, order=1, delay=1, n_bootstrap=1000, seed=42)
print('--- US GDP Growth ---')
print(f'Estatistica sup-LM: {hansen_gdp.statistic:.4f}')
print(f'P-valor (bootstrap, B=1000): {hansen_gdp.pvalue:.4f}')
print(f'Detalhe: {hansen_gdp.detail}')
if hansen_gdp.pvalue < 0.05:
    print('=> Rejeita H0 a 5%: threshold significativo')
else:
    print('=> Nao rejeita H0 a 5%: threshold nao significativo')
print()

# S&P500 Returns
hansen_sp = hansen_threshold_test(y_sp, order=1, delay=1, n_bootstrap=1000, seed=42)
print('--- S&P500 Returns ---')
print(f'Estatistica sup-LM: {hansen_sp.statistic:.4f}')
print(f'P-valor (bootstrap, B=1000): {hansen_sp.pvalue:.4f}')
if hansen_sp.pvalue < 0.05:
    print('=> Rejeita H0 a 5%: threshold significativo')
else:
    print('=> Nao rejeita H0 a 5%: threshold nao significativo')

## 4. Sequencia de testes de Terasvirta (1994): LSTAR vs ESTAR

Apos rejeitar linearidade, precisamos escolher entre **LSTAR** e **ESTAR**.
Terasvirta (1994) propoe uma sequencia de testes baseada na regressao auxiliar
do teste LM-STAR.

A regressao auxiliar inclui termos com $s_t$, $s_t^2$ e $s_t^3$:
$$
y_t = \beta_0 + \beta_1 y_{t-1} + \delta_1 y_{t-1} s_t + \eta_1 y_{t-1} s_t^2 + \psi_1 y_{t-1} s_t^3 + e_t
$$

Os testes parciais sao:
- $H_{04}$: $\psi_j = 0$ (termos cubicos)
- $H_{03}$: $\eta_j = 0 | \psi_j = 0$ (termos quadraticos)
- $H_{02}$: $\delta_j = 0 | \eta_j = \psi_j = 0$ (termos lineares em $s_t$)

**Regra de decisao de Terasvirta:**
1. Se p-valor de $H_{04}$ < p-valor de $H_{03}$ $\Rightarrow$ **LSTAR**
2. Se p-valor de $H_{03}$ < p-valor de $H_{04}$ $\Rightarrow$ **ESTAR**

Intuitivamente:
- Termos impares ($s_t$, $s_t^3$) capturam **assimetria** $\to$ LSTAR
- Termos pares ($s_t^2$) capturam **simetria** $\to$ ESTAR

In [ ]:
# TODO: Execute a sequencia de testes para escolher entre LSTAR e ESTAR

print('=== Sequencia de Testes de Terasvirta (1994) ===')
print()

# GDP Growth
tt_gdp = transition_type_test(y_gdp, order=1, delay=1)
print('--- US GDP Growth ---')
print(f'Modelo recomendado: {tt_gdp["recommended"]}')
print(f'F2 (H02): {tt_gdp["F2"]:.4f}, p-valor: {tt_gdp["p2"]:.4f}')
print(f'F3 (H03): {tt_gdp["F3"]:.4f}, p-valor: {tt_gdp["p3"]:.4f}')
print(f'F4 (H04): {tt_gdp["F4"]:.4f}, p-valor: {tt_gdp["p4"]:.4f}')
print(f'Detalhe: {tt_gdp["detail"]}')
print()

# S&P500 Returns
tt_sp = transition_type_test(y_sp, order=1, delay=1)
print('--- S&P500 Returns ---')
print(f'Modelo recomendado: {tt_sp["recommended"]}')
print(f'F2 (H02): {tt_sp["F2"]:.4f}, p-valor: {tt_sp["p2"]:.4f}')
print(f'F3 (H03): {tt_sp["F3"]:.4f}, p-valor: {tt_sp["p3"]:.4f}')
print(f'F4 (H04): {tt_sp["F4"]:.4f}, p-valor: {tt_sp["p4"]:.4f}')
print(f'Detalhe: {tt_sp["detail"]}')
print()

# Resumo
print('=== Resumo ===')
print(f'GDP Growth: sequencia recomenda {tt_gdp["recommended"]}')
print(f'S&P500: sequencia recomenda {tt_sp["recommended"]}')

## 5. Poder dos testes - Simulacao Monte Carlo

Vamos avaliar o **poder** dos testes de linearidade usando simulacao Monte Carlo.
Para cada DGP (Data Generating Process), simulamos $M$ series e calculamos
a taxa de rejeicao de $H_0$ a 5% de significancia.

DGPs considerados:
1. **AR(1) linear** (tamanho do teste - deveria rejeitar ~5%)
2. **SETAR(2)** com threshold abrupto
3. **LSTAR** com $\gamma$ moderado
4. **ESTAR** com $\gamma$ moderado

In [ ]:
# TODO: Simule poder dos testes sob diferentes DGPs

np.random.seed(42)
M = 200       # Numero de replicas Monte Carlo
T = 300       # Tamanho de cada serie
alpha = 0.05  # Nivel de significancia

def simulate_ar1(T, phi0=0.5, phi1=0.3, sigma=1.0):
    """Simular AR(1) linear."""
    y = np.zeros(T + 50)
    for t in range(1, len(y)):
        y[t] = phi0 + phi1 * y[t-1] + sigma * np.random.randn()
    return y[50:]  # descartar burn-in

def simulate_setar2(T, phi01=0.0, phi11=0.8, phi02=1.0, phi12=-0.5, c=0.5, sigma=1.0):
    """Simular SETAR(2) com threshold abrupto."""
    y = np.zeros(T + 50)
    for t in range(1, len(y)):
        if y[t-1] <= c:
            y[t] = phi01 + phi11 * y[t-1] + sigma * np.random.randn()
        else:
            y[t] = phi02 + phi12 * y[t-1] + sigma * np.random.randn()
    return y[50:]

def simulate_lstar(T, phi01=0.5, phi11=0.3, phi02=-0.5, phi12=-0.3, gamma=3, c=0, sigma=1.0):
    """Simular LSTAR."""
    y = np.zeros(T + 50)
    for t in range(1, len(y)):
        G = 1 / (1 + np.exp(-gamma * (y[t-1] - c)))
        y[t] = (phi01 + phi11 * y[t-1]) * (1 - G) + (phi02 + phi12 * y[t-1]) * G + sigma * np.random.randn()
    return y[50:]

def simulate_estar(T, phi01=0.5, phi11=0.3, phi02=-0.5, phi12=-0.3, gamma=3, c=0, sigma=1.0):
    """Simular ESTAR."""
    y = np.zeros(T + 50)
    for t in range(1, len(y)):
        G = 1 - np.exp(-gamma * (y[t-1] - c)**2)
        y[t] = (phi01 + phi11 * y[t-1]) * (1 - G) + (phi02 + phi12 * y[t-1]) * G + sigma * np.random.randn()
    return y[50:]

# Simulacao Monte Carlo
dgps = {
    'AR(1) linear': simulate_ar1,
    'SETAR(2)': simulate_setar2,
    'LSTAR': simulate_lstar,
    'ESTAR': simulate_estar,
}

tests = {
    'LM-STAR': lambda y: linearity_test(y, order=1, delay=1).pvalue,
    'Tsay': lambda y: tsay_test(y, order=1, delay=1).pvalue,
}

results = {dgp: {test: 0 for test in tests} for dgp in dgps}

for dgp_name, dgp_func in dgps.items():
    for _m in range(M):
        y_sim = dgp_func(T)
        for test_name, test_func in tests.items():
            try:
                pval = test_func(y_sim)
                if pval < alpha:
                    results[dgp_name][test_name] += 1
            except Exception:
                pass

# Converter para taxas de rejeicao
for dgp_name in results:
    for test_name in results[dgp_name]:
        results[dgp_name][test_name] /= M

# Exibir resultados
print(f'Simulacao Monte Carlo: M={M} replicas, T={T}, alpha={alpha}')
print()
df_results = pd.DataFrame(results).T
df_results.columns = [f'Rejeicao {t}' for t in tests]
print(df_results.to_string())
print()
print('Nota: AR(1) mostra tamanho empirico (deve ser ~5%)')
print('      DGPs nao-lineares mostram poder do teste')

## Visualizacao do poder dos testes

In [ ]:
# Grafico de barras com poder dos testes
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(dgps))
width = 0.35

bars1 = ax.bar(x - width/2, [results[d]['LM-STAR'] for d in dgps], width,
               label='LM-STAR', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, [results[d]['Tsay'] for d in dgps], width,
               label='Tsay', color='coral', alpha=0.8)

ax.axhline(alpha, color='red', linestyle='--', linewidth=1.5, label=f'Nivel nominal ({alpha})')
ax.set_xlabel('DGP', fontsize=12)
ax.set_ylabel('Taxa de rejeicao', fontsize=12)
ax.set_title(f'Poder dos Testes de Linearidade (M={M}, T={T})', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(list(dgps.keys()))
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)

# Adicionar valores nas barras
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/test_power_comparison.png', bbox_inches='tight')
plt.show()

## Resumo dos testes

| Teste | Alternativa | Metodo | Referencia |
|-------|------------|--------|------------|
| LM-STAR | STAR (LSTAR/ESTAR) | Expansao Taylor + F | Luukkonen et al. (1988) |
| Tsay | TAR (threshold abrupto) | Autorregressao arranjada | Tsay (1989) |
| Hansen | TAR (threshold) | sup-LM + bootstrap | Hansen (1996) |
| Terasvirta | LSTAR vs ESTAR | Sequencia de F parciais | Terasvirta (1994) |

**Estrategia recomendada:**
1. Testar linearidade com **LM-STAR** e **Tsay**
2. Se rejeitar, usar **sequencia de Terasvirta** para escolher LSTAR vs ESTAR
3. Se necessario, confirmar com **teste de Hansen** (bootstrap, mais robusto)
4. Estimar o modelo escolhido e validar com diagnosticos residuais

**Pontos-chave:**
- O teste LM-STAR tem bom poder contra alternativas STAR
- O teste de Tsay e mais potente contra TAR abrupto
- O teste de Hansen e o mais robusto (bootstrap), mas computacionalmente mais caro
- A sequencia de Terasvirta e essencial para a escolha LSTAR vs ESTAR